# fedOPD · build runbook — forecaster + federated loop

Successor to the EDA (closed at tag `eda-v2`). Turns the frozen 125-trust panel into trained
federated forecasters with an honest verdict against the **F-27 bar (naive-t-1 ~12.6% sMAPE)**.

**Conventions (see `docs/BUILD_LOG.md`, `docs/OPS.md`):**
- **Notebook-only** build, sectioned by step (owner instruction — overrides the runbook's modules+CLI
  proposal, recorded as D-06). Code here is AI-authored under the owner's explicit override of §12.
- **Device: CPU by default.** CUDA is used only if >20 GB is free (checked read-only so the shared,
  busy A100 is never disturbed). Whole grid ~7 TFLOP = CPU-minutes.
- Steps run in order; each ends in a **GATE**. A red gate STOPS the line.
- Findings are `R-xx`; decisions continue from `D-06`. Every run writes `results/*.json`.

## Step 0 · Preflight
Verify the inherited canonical table (D-01), pick the device without disturbing the GPU, cap threads,
and stamp the environment. **GATE:** row/month/column asserts pass and `results/env.json` is written.

In [1]:
# Step 0 - preflight: env, device policy, inherited canonical table.
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
import sys, json, hashlib, platform, subprocess, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd, torch
torch.set_num_threads(2)

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "CLAUDE.md").exists())
RESULTS = ROOT / "results"; RESULTS.mkdir(exist_ok=True)

# Device: use CUDA only if >20 GB is free (checked read-only via nvidia-smi, so we never disturb the
# shared A100), else CPU. The whole grid is ~7 TFLOP -> CPU-minutes, so CPU blocks nothing.
def gpu_free_gb():
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=5)
        return int(r.stdout.strip().splitlines()[0]) / 1024
    except Exception:
        return None
_free = gpu_free_gb()
DEVICE = "cuda" if (_free is not None and _free > 20 and torch.cuda.is_available()) else "cpu"
print(f"device: {DEVICE}  (gpu free: {None if _free is None else round(_free, 1)} GB)")

# Inherited canonical table (D-01) - the ONLY input; never re-read the raw CSVs.
canon = pd.read_parquet(ROOT / "reports" / "tables" / "rtt_canonical.parquet")
EXPECT_COLS = ["date", "branch", "branch_name", "icb", "dept", "dept_name", "demand"]
assert len(canon) == 227344, f"row count {len(canon)} != 227344 - EDA/build disagree, STOP"
assert canon.date.nunique() == 96, "month count != 96"
assert list(canon.columns) == EXPECT_COLS, f"columns {list(canon.columns)}"
print(f"canonical OK: {len(canon):,} rows | {canon.date.nunique()} months "
      f"{canon.date.min().date()}..{canon.date.max().date()}")

def git_sha():
    try:
        return subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip()
    except Exception:
        return "?"

env = {"when": dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "host": platform.node(),
       "git": git_sha(), "python": sys.version.split()[0], "torch": torch.__version__,
       "numpy": np.__version__, "pandas": pd.__version__, "device": DEVICE,
       "gpu_free_gb": None if _free is None else round(_free, 1),
       "cpu_count": os.cpu_count(), "omp_threads": os.environ.get("OMP_NUM_THREADS"),
       "rows": int(len(canon)), "months": int(canon.date.nunique())}
(RESULTS / "env.json").write_text(json.dumps(env, indent=2))
print("GATE Step 0: PASS - asserts pass, env stamped ->", (RESULTS / "env.json"))

device: cpu  (gpu free: 6.6 GB)
canonical OK: 227,344 rows | 96 months 2018-04-01..2026-03-01
GATE Step 0: PASS - asserts pass, env stamped -> /home/garima/fedOPD/results/env.json


## Step 1 · Freeze the panel, roster, and eligible series
Apply D-03 (125-trust panel) and the F-28 roster decision (18 window-complete departments, excluding
the X 'Other' family - **D-07, pending supervisor Q5**). Filter to series that can yield >=24 train
windows and >=1 test window under window W=12 (D-02 validity). Freeze `results/panel.json` (hashed);
everything downstream reads this file. **GATE:** 125 trusts, series+drops logged, panel.json written.

In [2]:
# Step 1 - freeze panel, roster, and eval / train-only / dropped series -> results/panel.json
from collections import Counter
months = pd.Index(sorted(canon.date.unique()))
TRAIN_END = pd.Timestamp("2025-03-01"); W = 12; MIN_TRAIN_WINDOWS = 24
DEPTS_MODE = "stable18"; MIN_COVERAGE = 0.5                 # supervisor Q5 roster / Q4 threshold (parametrised)

pm = canon.groupby("branch").date.nunique()
panel = sorted(pm.index[pm == 96]); pset = set(panel)
variants = {"ge96_default": int((pm==96).sum()), "ge90": int((pm>=90).sum()), "ge84": int((pm>=84).sum())}
assert len(panel) == 125, f"panel {len(panel)} != 125"

# Roster (F-28): stable-18 = depts present all 96 months (excludes X01 36mo, X02-X06 60mo). D-07 default.
dept_months = canon.groupby("dept").date.nunique()
stable18 = sorted(dept_months.index[dept_months == 96])
roster_full = {"stable18": stable18, "all24": sorted(canon.dept.unique()),
               "no_x": sorted(d for d in canon.dept.unique() if not str(d).startswith("X"))}
cov = canon[canon.branch.isin(pset)].groupby("dept").branch.nunique() / len(panel)   # coverage among panel
roster = [d for d in roster_full[DEPTS_MODE] if cov.get(d, 0) >= MIN_COVERAGE]
roster_dropped = {d: round(float(cov.get(d, 0)), 3) for d in roster_full[DEPTS_MODE] if cov.get(d, 0) < MIN_COVERAGE}

# Window validity (D-02): window at target t valid iff months[t-W:t] present AND month t present.
tr_idx = [i for i, m in enumerate(months) if i >= W and m <= TRAIN_END]
te_idx = [i for i, m in enumerate(months) if m > TRAIN_END]
pres = (canon.assign(one=1).pivot_table("one", ["branch","dept"], "date", fill_value=0)
        .reindex(columns=months, fill_value=0).astype(bool))

# Three sets. A series with >=24 train windows but NO valid test window is NOT simply dropped: if it is
# still active (reports in a 2025-04+ month = interior gap) it is kept TRAIN-ONLY (it carries gradient
# signal to the shared model but cannot be evaluated); if it went stale (no report after 2025-03) it is
# dropped. Every method then trains on the identical set (eval + train-only) so comparisons hold.
eval_series, train_only, drops = [], [], []
for (b, d), row in pres.iterrows():
    if b not in pset or d not in roster:
        continue
    v = row.values
    tr = sum(bool(v[i]) and bool(v[i-W:i].all()) for i in tr_idx)
    te = sum(bool(v[i]) and bool(v[i-W:i].all()) for i in te_idx)
    if tr < MIN_TRAIN_WINDOWS:              drops.append((b, d, "few-train-windows"))
    elif te >= 1:                           eval_series.append([b, d, int(tr), int(te)])
    elif bool(v[te_idx].any()):             train_only.append([b, d, int(tr)])     # active interior-gap
    else:                                   drops.append((b, d, "stale-closed"))
reasons = Counter(r for _, _, r in drops)
train_series = [[b, d] for b, d, *_ in eval_series] + [[b, d] for b, d, _ in train_only]
client_trusts = sorted({b for b, d in train_series})
print(f"panel {len(panel)} trusts | roster {len(roster)}/{len(roster_full[DEPTS_MODE])} depts "
      f"| dropped-by-coverage {roster_dropped}")
print(f"eval {len(eval_series)} + train-only {len(train_only)} = {len(train_series)} training series "
      f"| {len(client_trusts)} client trusts | drops {dict(reasons)}")

panel_obj = {"created": env["when"], "git": env["git"], "panel_trusts": panel, "panel_variants": variants,
             "depts_mode": DEPTS_MODE, "roster": roster, "roster_dropped": roster_dropped,
             "min_coverage": MIN_COVERAGE, "window": W, "min_train_windows": MIN_TRAIN_WINDOWS,
             "train_end": str(TRAIN_END.date()), "test_months": [str(months[i].date()) for i in te_idx],
             "eval_series": eval_series, "train_only_series": train_only,
             "n_eval_series": len(eval_series), "n_train_series": len(train_series),
             "client_trusts": client_trusts, "n_clients": len(client_trusts), "drop_reasons": dict(reasons)}
panel_obj["hash"] = hashlib.sha256(json.dumps(panel_obj, sort_keys=True).encode()).hexdigest()[:16]
(RESULTS / "panel.json").write_text(json.dumps(panel_obj, indent=2))
ok = len(panel) == 125 and len(train_series) > 0 and (RESULTS / "panel.json").exists()
print("panel.json hash:", panel_obj["hash"]); print("GATE Step 1:", "PASS" if ok else "FAIL")

panel 125 trusts | roster 17/18 depts | dropped-by-coverage {'C_150': 0.36}
eval 1528 + train-only 16 = 1544 training series | 117 client trusts | drops {'few-train-windows': 137, 'stale-closed': 18}
panel.json hash: e3aa353ecef2cbcb
GATE Step 1: PASS


## Step 2 · Build the datasets
Per series: reindex to month-start, window W=12 (horizon 1). **Features (33 = 12 scaled lags + month
sin/cos + COVID flag + 2023-strike flag + 17 dept one-hots).** Per-series scalers fitted on TRAIN months
only (≤2025-03), **excluding outlier-flagged months** (D-05); std floored at 1e-6. A window is used only
if all 12 lags and the target are present (D-02) — no imputation. Training set = eval + train-only series;
evaluation set = eval series' test windows. **GATE: five leakage asserts must pass.**

In [3]:
# Step 2 - windows + per-series scalers (TRAIN-only, excl. outlier-flagged months) -> results/dataset_stats.json
reg = pd.read_csv(ROOT/"reports"/"tables"/"07_outlier_register.csv", parse_dates=["date"])
flagged = set(zip(reg.branch, reg.dept, reg.date))         # months excluded from scaler fitting (D-05)

dem = canon.pivot_table("demand", ["branch","dept"], "date").reindex(columns=months)   # NaN where absent
mo = np.array([m.month for m in months])
MSIN = np.sin(2*np.pi*mo/12); MCOS = np.cos(2*np.pi*mo/12)
COVID  = ((months>=pd.Timestamp("2020-03-01"))&(months<=pd.Timestamp("2021-03-01"))).astype(float)
STRIKE = ((months>=pd.Timestamp("2022-12-01"))&(months<=pd.Timestamp("2023-12-01"))).astype(float)
ridx = {d:i for i,d in enumerate(roster)}; NR = len(roster)
N_FEATURES = W + 4 + NR
tr_targets = [i for i in range(W,len(months)) if months[i]<=TRAIN_END]
te_targets = [i for i in range(W,len(months)) if months[i]>TRAIN_END]

def fit_scaler(b, d):                                      # per-series mean/std, TRAIN only, excl. flagged
    row = dem.loc[(b,d)].values
    keep = [row[i] for i in range(len(months))
            if months[i] <= TRAIN_END and not np.isnan(row[i]) and (b,d,months[i]) not in flagged]
    if not keep: return 0.0, 1.0
    a = np.array(keep); return float(a.mean()), max(float(a.std()), 1e-6)

def make_row(row, t, mean, sd, oh):
    return np.concatenate([(row[t-W:t]-mean)/sd, [MSIN[t], MCOS[t], COVID[t], STRIKE[t]], oh])

scalers = {}
Xtr, ytr, tr_owner = [], [], []
Xte, yte, te_meta = [], [], []
for b, d in ([tuple(s[:2]) for s in eval_series] + [tuple(s[:2]) for s in train_only]):
    row = dem.loc[(b,d)].values; mean, sd = fit_scaler(b,d); scalers[(b,d)] = (mean, sd)
    oh = np.zeros(NR); oh[ridx[d]] = 1.0
    for t in tr_targets:
        if np.isnan(row[t]) or np.isnan(row[t-W:t]).any(): continue
        Xtr.append(make_row(row,t,mean,sd,oh)); ytr.append((row[t]-mean)/sd); tr_owner.append(b)
for s in eval_series:
    b, d = s[0], s[1]; row = dem.loc[(b,d)].values; mean, sd = scalers[(b,d)]
    oh = np.zeros(NR); oh[ridx[d]] = 1.0
    for t in te_targets:
        if np.isnan(row[t]) or np.isnan(row[t-W:t]).any(): continue
        Xte.append(make_row(row,t,mean,sd,oh)); yte.append((row[t]-mean)/sd)
        te_meta.append((b, d, str(months[t].date()), mean, sd, float(row[t])))
Xtr, ytr = np.asarray(Xtr, np.float32), np.asarray(ytr, np.float32)
Xte, yte = np.asarray(Xte, np.float32), np.asarray(yte, np.float32)
te_df = pd.DataFrame(te_meta, columns=["branch","dept","month","mean","sd","actual"])

# ---- GATE: five leakage asserts ----
tmonths = pd.to_datetime(te_df.month)
assert bool(((tmonths>=pd.Timestamp("2025-04-01"))&(tmonths<=pd.Timestamp("2026-03-01"))).all()), "test month out of range"
assert not (np.isnan(Xtr).any() or np.isnan(ytr).any() or np.isnan(Xte).any() or np.isnan(yte).any()), "NaN in X/y"
tr_trusts, te_trusts = set(tr_owner), set(te_df.branch)
assert te_trusts.issubset(tr_trusts) and len(te_trusts) > 0, "a test trust does not train"
for (b,d) in scalers:                                      # scaler fit never touched a test month
    row = dem.loc[(b,d)].values
    fit_months = [months[i] for i in range(len(months))
                  if months[i] <= TRAIN_END and not np.isnan(row[i]) and (b,d,months[i]) not in flagged]
    assert (not fit_months) or max(fit_months) <= TRAIN_END, f"scaler leak {b} {d}"
assert len(ytr) > 0 and len(yte) > 0

stats = {"n_features": N_FEATURES, "window": W, "horizon": 1, "roster_size": NR,
         "n_train_windows": int(len(ytr)), "n_test_windows": int(len(yte)),
         "n_train_series": len(eval_series)+len(train_only), "n_eval_series": len(eval_series),
         "n_train_only_series": len(train_only), "n_clients": len(tr_trusts), "n_test_clients": len(te_trusts),
         "train_windows_per_client_min": int(pd.Series(tr_owner).value_counts().min()),
         "feature_layout": f"{W} scaled lags + [sin,cos,covid,strike] + {NR} dept one-hot"}
(RESULTS/"dataset_stats.json").write_text(json.dumps(stats, indent=2))
print("leakage asserts: all 5 PASS")
print(f"X_train {Xtr.shape} | X_test {Xte.shape} | {N_FEATURES} features | {len(tr_trusts)} clients "
      f"| min train-windows/client {stats['train_windows_per_client_min']}")
print("GATE Step 2: PASS ->", RESULTS/"dataset_stats.json")

leakage asserts: all 5 PASS
X_train (108065, 33) | X_test (18191, 33) | 33 features | 117 clients | min train-windows/client 54
GATE Step 2: PASS -> /home/garima/fedOPD/results/dataset_stats.json
